# LUNES 

<b> probabilidad | ¿Qué nos dice la distribución empírica sobre el riesgo?
Modalidad: Código. </b>

Imagina que tenemos tres clases de activos: acciones, bonos y oro.

Primero construiremos 252 observaciones sintéticas de retornos diarios.

```python
import numpy as np
import pandas as pd

rng = np.random.default_rng(117)

T = 252
z = rng.standard_normal((T, 3))

equity = 0.00045 + 0.011 * z[:, 0]

bonds = (
    0.00015
    + 0.003 * (
        0.2 * z[:, 0]
        + np.sqrt(0.96) * z[:, 1]
    )
)

gold = (
    0.00020
    + 0.008 * (
```

Todos los retornos están expresados en formato decimal.

No son observaciones reales de mercado.

**Parte A — Construye dos portafolios**

Portfolio A:

$$w_A = \begin{bmatrix} 0.60 \\ 0.25 \\ 0.15 \end{bmatrix}$$


Portfolio B:

$$w_B = \begin{bmatrix} 0.40 \\ 0.35 \\ 0.25 \end{bmatrix}.$$

Supondremos pesos constantes mediante rebalanceo diario idealizado, sin costes.

Calcula:

$$R_{P,t} = w^\top R_t.$$

Para cada portafolio estima:

$$\hat{\mu} = \frac{1}{T} \sum_t R_{P,t}$$

$$\hat{\sigma} = \text{Std}(R_P).$$

Anualiza:

$$\mu_{\text{ann}} = 252\hat{\mu}$$

$$\sigma_{\text{ann}} = \sqrt{252}\hat{\sigma}.$$

La media anualizada es una aproximación aritmética, no el rendimiento compuesto anual realizado.



**Parte B — Riesgo de cola**

Obtén el cuantil empírico del 5%:

$$q_{0.05}(R_P).$$

Define el VaR histórico al 95%, expresado como pérdida positiva:

$$VaR_{95\%} = -q_{0.05}(R_P).$$

Después estima:

$$ES_{95\%} = -E[R_P \mid R_P \leq q_{0.05}].$$

Finalmente calcula el máximo drawdown a partir de:

$$W_t = \prod_{s=1}^t (1 + R_{P,s}),$$

incluyendo $W_0 = 1$.

Construye esta tabla:

| Métrica | Portfolio A | Portfolio B |
|---|---:|---:|
| Annualized Mean Return | | |
| Annualized Volatility | | |
| Historical VaR 95% | | |
| Historical ES 95% | | |
| Maximum Drawdown | | |

**Parte C — Interpretación**

Responde tres preguntas.

1. ¿Reducir volatilidad implica necesariamente reducir Expected Shortfall?

2. ¿Por qué un evento de estrés puede afectar el drawdown de manera distinta a la volatilidad muestral?

3. Si el Portfolio B presenta mejores métricas históricas de riesgo, ¿puedes concluir que será superior fuera de muestra?

Considera que disponemos de una muestra relativamente pequeña para estudiar eventos extremos y que los parámetros de la simulación son conocidos.

**Entregable:** Un DataFrame comparativo y tres conclusiones.

No optimices los pesos. Los dos portafolios fueron definidos antes de evaluar sus resultados.



In [6]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(117)

T = 252
z = rng.standard_normal((T, 3))

equity = 0.00045 + 0.011 * z[:, 0]

bonds = (
    0.00015
    + 0.003 * (
        0.2 * z[:, 0]
        + np.sqrt(0.96) * z[:, 1]
    )
)

gold = (
    0.00020
    + 0.008 * (
        0.1 * z[:, 0]
        + np.sqrt(0.99) * z[:, 2]
    )
)

R = np.column_stack([equity, bonds, gold])

# Escenario de estrés sintético
R[180] = [-0.06, -0.012, 0.016]

# Construccion de los 2 portafolios 
w_A = np.array([0.60, 0.25, 0.15])
w_B = np.array([0.40, 0.35, 0.25])
# supondremnos pesos constantes mediante rebalanaceo diario, sin costes 
def portfolio_calculos(R, w):
    # Retorno diario del portfolio: 252 observaciones
    portfolio_returns = R @ w

    # Parte A
    mu_daily = portfolio_returns.mean()
    sigma_daily = portfolio_returns.std(ddof=1)

    mu_ann = 252 * mu_daily
    sigma_ann = np.sqrt(252) * sigma_daily

    # Parte B: riesgo de cola
    q_05 = np.quantile(portfolio_returns, 0.05)

    # Pérdida positiva
    var_95 = -q_05

    tail_returns = portfolio_returns[portfolio_returns <= q_05]
    es_95 = -tail_returns.mean()

    # Wealth con W_0 = 1
    wealth = np.r_[1.0, np.cumprod(1 + portfolio_returns)]
    peak = np.maximum.accumulate(wealth)
    drawdown = wealth / peak - 1
    max_drawdown = drawdown.min()

    return {
        "Annualized Mean Return": mu_ann,
        "Annualized Volatility": sigma_ann,
        "Historical VaR 95%": var_95,
        "Historical ES 95%": es_95,
        "Maximum Drawdown": max_drawdown
    }

metricas_A = portfolio_calculos(R, w_A)
metricas_B = portfolio_calculos(R, w_B)

df_metricas = pd.DataFrame({
    "Portfolio A": metricas_A,
    "Portfolio B": metricas_B
})

df_metricas

,Portfolio A,Portfolio B
Annualized Mean Return,0.182980,0.147736
Annualized Volatility,0.116894,0.084665
Historical VaR 95%,0.010863,0.008082
Historical ES 95%,0.015766,0.011493
Maximum Drawdown,-0.059917,-0.047482


1. ¿Reducir volatilidad implica necesariamente reducir Expected Shortfall?
- R: 
> el Expected Shortfall mide exclusivamente la pérdida media dentro del peor 5% de los retornos.
Un portfolio puede tener menor volatilidad promedio, pero seguir expuesto a pérdidas extremas por asimetría negativa, colas pesadas, saltos de precio o correlaciones que aumentan durante estrés. En ese caso, su ES puede ser igual o incluso mayor que el de un portfolio más volátil.

2. ¿Por qué un evento de estrés puede afectar el drawdown de manera distinta a la volatilidad muestral?

3. Si el Portfolio B presenta mejores métricas históricas de riesgo, ¿puedes concluir que será superior fuera de muestra?

Considera que disponemos de una muestra relativamente pequeña para estudiar eventos extremos y que los parámetros de la simulación son conocidos.